# Module 307 — Safe Function Calling

This offline example models a shipping-support tool that can look up an order. The model may propose a tool name and arguments, but application code chooses from a fixed registry and checks the authenticated user's access. All data is fictional and local.

**Important:** never turn a model response into executable code with `eval` or `exec`. A tool-call request is a proposal, not proof of permission.

## 1. Create fictional order records

Each order includes an owner ID. In a real application, the authenticated user ID comes from trusted server-side session or identity data—not from a prompt or a model-generated argument.

In [ ]:
ORDERS = {
    1337: {'owner_id': 'user-7', 'status': 'in transit'},
    2001: {'owner_id': 'user-9', 'status': 'delivered'},
}
authenticated_user_id = 'user-7'

## 2. Write a function that enforces authorization

This tool accepts an integer order ID, confirms that the record exists, and checks its owner before returning anything. A production implementation would also handle database errors and use a parameterized query if its records came from SQL.

In [ ]:
def get_order_status(order_id, user_id):
    # Use an exact integer check so unexpected values are rejected.
    if type(order_id) is not int:
        raise ValueError('order_id must be an integer')

    order = ORDERS.get(order_id)
    if order is None:
        raise LookupError('order not found')

    # Authorization is checked in application code using trusted identity.
    if order['owner_id'] != user_id:
        raise PermissionError('user cannot access this order')

    return order['status']

get_order_status(1337, authenticated_user_id)

## 3. Dispatch only approved function names

The registry is written by application developers. The model cannot add a new tool just by returning a new name. The dispatcher also rejects unexpected argument keys, then invokes the approved function. It never interprets the model's text as Python source.

In [ ]:
TOOL_REGISTRY = {
    'get_order_status': get_order_status,
}

def dispatch_tool(tool_name, arguments, authenticated_user_id):
    function = TOOL_REGISTRY.get(tool_name)
    if function is None:
        raise ValueError('tool is not approved')
    if not isinstance(arguments, dict) or set(arguments) != {'order_id'}:
        raise ValueError('unexpected tool arguments')
    return function(arguments['order_id'], authenticated_user_id)

# A model-proposed call is still checked by the application.
dispatch_tool('get_order_status', {'order_id': 1337}, authenticated_user_id)

## 4. Verify access is tied to the real user

Order 2001 belongs to `user-9`, so the authenticated `user-7` should be denied even if a prompt asks the model to retrieve it or claims the caller is an administrator. The claim in the prompt does not change the trusted identity passed by the application.

In [ ]:
try:
    dispatch_tool('get_order_status', {'order_id': 2001}, authenticated_user_id)
except PermissionError as error:
    print('Authorization denied:', error)

try:
    dispatch_tool('system_check', {'command': 'anything'}, authenticated_user_id)
except ValueError as error:
    print('Unknown tool denied:', error)

## COAE quick review

- **Insecure implementation:** model output reaches `eval` or `exec`.
- **Excessive agency:** tools or permissions exceed the user's need or authority.
- **Vulnerable function:** an allowed tool contains a bug such as SQL injection.
- Use a fixed registry, validate arguments, authorize using trusted identity, and secure each function's downstream operations.
- Prompt injection can influence a tool proposal; it does not authenticate the user or justify executing the proposal.

This worksheet makes no network requests and executes no model-generated code.